In [0]:
%python
from datetime import datetime
from dateutil import parser
import json

# Define widgets
dbutils.widgets.text("last_ingest_time", "", "Ruta last_ingest_time_date")
dbutils.widgets.text("last_ingest_time_date", "", "Date last_ingest_time_date")

# Obtiene valores de los widgets
last_ingest_time = dbutils.widgets.get("last_ingest_time")
last_ingest_time_date = dbutils.widgets.get("last_ingest_time_date")

try:
    last_ingest_time_dt = parser.isoparse(last_ingest_time_date)
except Exception as e:
    raise Exception(f"Error al parsear last_ingest_time_date: {last_ingest_time_date}. Detalle: {str(e)}")

try:
    current_content = dbutils.fs.head(last_ingest_time)
    current_lines = current_content.strip().split("\n")
    current_date_str = current_lines[1].replace(";", "").strip()
    current_date = parser.isoparse(current_date_str)
    print(f"📄 Fecha actual en el archivo: {current_date_str}")
except Exception as e:
    print(f"No se pudo leer la fecha actual del archivo. Se actualizará de todos modos. Detalle: {str(e)}")
    current_date = None

# Verifica si se actualiza el last ingest time
if current_date is None or last_ingest_time_dt > current_date:
    try:
        new_content = f"timestamp;\n{last_ingest_time_dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')};"
        dbutils.fs.put(last_ingest_time, new_content, overwrite=True)
        print(f"Archivo {last_ingest_time} actualizado con la nueva fecha: {last_ingest_time_dt}")
    except Exception as e:
        print(f"Error al actualizar el archivo {last_ingest_time}: {str(e)}")
        dbutils.notebook.exit("Error: No se pudo actualizar el archivo")
else:
    print(f"La fecha {last_ingest_time_dt} NO es mayor que la actual. No se actualiza el archivo.")

try:
    content = dbutils.fs.head(last_ingest_time)
    print(f"📄 Contenido actual del archivo:\n{content}")
except Exception as e:
    print(f"Error al leer el archivo {last_ingest_time}: {str(e)}")


In [0]:
%python

#Librerías
from pyspark.sql.functions import to_date, col, regexp_replace
from pyspark.sql.types import StringType

try:
    # Definir widgets sin valores por defecto
    dbutils.widgets.text("file_path", "", "")
    dbutils.widgets.text("campo_particion", "", "")
    dbutils.widgets.text("base_temp_path", "")
    dbutils.widgets.text("base_final_path", "")
    dbutils.widgets.text("excel_range", "'Sheet0'!A3:AX1048576")
    dbutils.widgets.text("filename_excel", "")

    # Obtener los valores de los widgets
    file_path = dbutils.widgets.get("file_path")
    base_temp_path = dbutils.widgets.get("base_temp_path")
    base_final_path = dbutils.widgets.get("base_final_path")
    campo_particion = dbutils.widgets.get("campo_particion")
    excel_range = dbutils.widgets.get("excel_range")
    filename_excel = dbutils.widgets.get("filename_excel")

    # Lee archivo Excel
    df = (spark.read
          .format("com.crealytics.spark.excel")
          .option("header", "true")
          .option("dataAddress", excel_range)
          .option("maxRowsInMemory", 10000)
          .option("treatEmptyValuesAsNulls", "true")
          .load(file_path + filename_excel))

    # Valida si viene vacío
    if df.limit(1).count() == 0:
        dbutils.notebook.exit("El archivo no contiene registros.")

    # Convierte la columna de partición a fecha
    df = df.withColumn(campo_particion, to_date(col(campo_particion), "M/d/yy"))

    # Limpia saltos de línea en columnas string
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, regexp_replace(col(c), r"[\r\n]+", " "))

    unique_dates = [row[campo_particion] for row in df.select(campo_particion).where(col(campo_particion).isNotNull()).distinct().collect()]

    for date_val in unique_dates:
        formatted_date = date_val.strftime("%d%m%Y")
        df_filtered = df.filter(col(campo_particion) == date_val)

        temp_path = f"{base_temp_path}/resp_call_center_{formatted_date}"
        final_path = f"{base_final_path}/resp_call_center_{formatted_date}.csv"

        df_filtered.repartition(1) \
            .write.mode("overwrite") \
            .option("header", "true") \
            .option("delimiter", "~") \
            .option("quote", "\"") \
            .option("escape", "\"") \
            .csv(temp_path)

        files = dbutils.fs.ls(temp_path)
        csv_file = [f.path for f in files if f.path.endswith(".csv")][0]

        dbutils.fs.mv(csv_file, final_path, True)
        dbutils.fs.rm(temp_path, True)

    full_path = file_path + filename_excel
    dbutils.fs.rm(full_path, True)
    dbutils.notebook.exit("Archivo procesado correctamente")

except Exception as e:
    raise Exception(f"Error procesando el archivo: {str(e)}")